In [4]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

In [8]:
df = pd.read_csv('../data/processed/telco_features.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,No_internet_service,No_phone_service
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,1,0,0,1
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,1,0,0,0,1,0,0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,1,0,0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,1,0,0,0,0,0,1
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,1,0,0,0


## Class Imbalance


~27% churners. With imbalance, **accuracy is misleading** (predicting "no churn" for everyone is ~73% accurate and useless). Class weighting + threshold tuning is enough here — no aggressive oversampling needed.

In [12]:
df['Churn'].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

### Why recall matters most in churn

The costs are **asymmetric**:
- **False Negative** — predict "won't churn" but they leave → you miss the chance to retain them (expensive: lost customer).
- **False Positive** — predict "will churn" but they stay → you spend a retention offer unnecessarily (cheap).

Missing a churner usually costs more than a wasted offer → prioritize **recall**.


Typical priority:
- Retention campaigns **cheap** → maximize recall (catch every possible churner).
- Retention campaigns **expensive** → balance precision/recall via F1.


In [13]:
# Train/test split (stratified to preserve class ratio)
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

THRESHOLD = 0.3  # lower than 0.5 to boost recall (tuned per model below)

## RandomForest

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.892     0.755     0.818      1033
           1      0.525     0.749     0.617       374

    accuracy                          0.753      1407
   macro avg      0.709     0.752     0.718      1407
weighted avg      0.795     0.753     0.765      1407

